# pybioclip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imageomics/evolution-workshop-2026/blob/main/docs/tutorials/notebooks/pybioclip.ipynb)

## Learning objectives

By the end of this tutorial, you will be able to:

1. Install `pybioclip` and run it on example images, from both the command line and Python.
2. Classify an organism against the full Tree of Life and interpret the ranked predictions.
3. Score an image against your own set of custom labels.
4. Generate image embeddings (feature vectors) for downstream tasks such as similarity search and clustering.

## Prerequisites

- **Python:** >= 3.10 (Colab satisfies this by default).
- **Packages:** `pybioclip` (installed below). Pulls in `torch`, `torchvision`, `open_clip_torch`.
- **Data:** two example images downloaded in the Setup step, so no data prep is required.
- **Prior knowledge:** basic Python. No machine-learning background needed.

## Background

[BioCLIP](https://imageomics.github.io/bioclip-ecosystem/pages/models.html) is a family of vision foundation models trained on the [TreeOfLife](https://imageomics.github.io/bioclip-ecosystem/pages/data.html) datasets to understand images of organisms across the tree of life. [`pybioclip`](https://imageomics.github.io/pybioclip/) is a small Python library (and CLI) that wraps these models so you can classify images and extract embeddings in a few lines, with no model-loading or tensor wrangling required. It uses BioCLIP 2 by default.

Two ideas you'll use below:

- **Classification** turns an image into a ranked list of labels with scores. Against the Tree of Life you get taxonomic predictions from kingdom down to species; with custom labels you get scores for terms *you* supply.
- **Embeddings** turn an image into a fixed-length vector that captures its visual content. Similar organisms land near each other in this space, which is the basis for similarity search, clustering, and downstream classifiers.

## Setup

**Use a GPU runtime.** Before running anything, select Runtime, then Change runtime type, then T4 GPU for much faster inference.

Install `pybioclip` and download two example images from the BioCLIP demo: a brown bear and a domestic cat.

In [ ]:
%pip install -q pybioclip

In [ ]:
# Retrieve example images.
!wget -q "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Ursus-arctos.jpeg"
!wget -q "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Felis-catus.jpeg"

In [ ]:
# Preview the images.
from PIL import Image
from IPython.display import display

examples = ["Ursus-arctos.jpeg", "Felis-catus.jpeg"]

for name in examples:
  img = Image.open(name)
  print(name)
  display(img.resize((224,224)))

## Try it from the command line

Installing `pybioclip` gives you a `bioclip` command-line interface (CLI) to predict image labels and create embeddings of images.

**To open a terminal in Colab**, click the Terminal button at the bottom-left of the window. 

> 📝You can delete `sample_data/` that was placed there by Google.

Then run the commands below at the terminal prompt, which is:
```bash
/content# 
```

**Predict labels for the contents of an image using all labels in the TreeOfLife dataset.**

Predict the five most confident labels for the cat image.

```bash
bioclip predict --device cuda Felis-catus.jpeg
```

The warning we see can be suppressed. Tell the package that we do not need to check Hugging Face on each command to confirm we have the most up-to-date files cached.

```bash
export HF_HUB_OFFLINE=1
```

Use `--k` to see any number of ranked candidates (5 by default).

```bash
bioclip predict --device cuda Ursus-arctos.jpeg --k 1
```

See all of the `bioclip predict` options with the help menu:

```bash
bioclip predict --help
```

**Predict down to a certain taxonomic rank.**

```bash
bioclip predict --device cuda Ursus-arctos.jpeg --rank family
```

**Classify with your own labels.**

Pass a comma-separated list to `--cls`. Here we ask which of `bear`, `cat`, or `bat` matches each.

```bash
bioclip predict --device cuda --cls bear,cat,bat *.jpeg
```

> When using custom labels, each label needs to be embedded by the text encoder.

**Where do the open-ended labels come from?** Tree of Life prediction scores an image against every taxon in BioCLIP's built-in label set. List that set and save it to a file:

```bash
bioclip list-tol-taxa > tol-taxa.csv
```

Each row is one taxon, with its full taxonomy (down to `species`) and a `common_name`. Find the rows for the cat genus:

```bash
grep ",Felis," tol-taxa.csv
```

**Restrict prediction to a subset of taxa** with `--subset`, which can sharpen results by ignoring irrelevant taxa. Pass a CSV whose first column is a rank; here we list a few `Felis` species:

```bash
cat > species.csv <<EOF
species
Felis catus
Felis silvestris
Felis chaus
EOF

bioclip predict --device cuda --subset species.csv Felis-catus.jpeg
```

**Generate embeddings.** Embed one or more images. By default the results print to the screen; redirect them to a file with `>`:

```bash
bioclip embed --device cuda *.jpeg > embeddings.json
```

**Use a different model** to predict or generate embeddings. List the available models, then pass one with `--model`:

```bash
bioclip list-models
bioclip predict --device cuda --model hf-hub:imageomics/bioclip Ursus-arctos.jpeg
```

Run `bioclip --help`, or `bioclip predict --help`, to see all available options.

## Now in Python

The `bioclip` CLI just wraps a Python package you can call directly. Working in Python hands you the predictions as objects to sort, filter, or plot. Here are the same operations in code.

## Classify against the Tree of Life

`TreeOfLifeClassifier` ranks an image against BioCLIP's full taxonomy. We pass `device` so it runs on the GPU when one is available. Pick a `Rank` to predict at; each result is a dict of taxonomic fields plus a `score`.

In [ ]:
import torch
from bioclip import TreeOfLifeClassifier, Rank

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = TreeOfLifeClassifier(device=device)
predictions = classifier.predict("Ursus-arctos.jpeg", Rank.SPECIES)

for p in predictions:
    print(p["species"], p["common_name"], p["score"])

That first prediction downloaded and cached BioCLIP's weights. Now we can tell the Hugging Face Hub to work offline, which reuses the cached files and skips the per-command checks (and the not-signed-in warning) for the rest of the notebook.

In [ ]:
import os
import huggingface_hub

# The weights are cached now, so use them directly and stop contacting the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
huggingface_hub.constants.HF_HUB_OFFLINE = True

**Predict at a coarser rank:**

In [ ]:
predictions = classifier.predict("Ursus-arctos.jpeg", Rank.FAMILY, k=1)

for p in predictions:
    print(p["family"], p["score"])

## Classify with your own labels

`CustomLabelsClassifier` scores an image against a list of labels you supply, instead of the whole taxonomy. Each result is a dict with `classification` and `score`. Here we ask which of `bear`, `cat`, or `bat` best matches the cat.

In [ ]:
from bioclip import CustomLabelsClassifier

classifier = CustomLabelsClassifier(["bear", "cat", "bat"], device=device)
for p in classifier.predict("Felis-catus.jpeg"):
    print(p["classification"], p["score"])

## Generate image embeddings

`create_image_features` returns one normalized feature vector per image as a `torch.Tensor`. Embeddings like these power similarity search, clustering, and lightweight downstream classifiers.

In [ ]:
from bioclip import TreeOfLifeClassifier
from PIL import Image

classifier = TreeOfLifeClassifier(device=device)
images = [Image.open(p).convert("RGB") for p in ["Ursus-arctos.jpeg", "Felis-catus.jpeg"]]
features = classifier.create_image_features(images, normalize=True)
print("features shape:", tuple(features.shape))  # (num_images, embedding_dim)

The vectors are L2-normalized, so the dot product of two of them is their cosine similarity, a quick measure of how visually alike two organisms are:

In [ ]:
bear, cat = features[0], features[1]
similarity = (bear @ cat).item()
print("cosine similarity (bear vs cat):", round(similarity, 4))

## Summary

You installed `pybioclip` and used BioCLIP 2 to (1) classify an image against the Tree of Life, (2) score it against custom labels, and (3) generate image embeddings and measure similarity, all without managing ML infrastructure.

**Next steps**

- [pybioclip documentation](https://imageomics.github.io/pybioclip/): full Python API and CLI reference.
- Other workshop tutorials at [Designing for Discovery](https://imageomics.github.io/evolution-workshop-2026/).